In [1]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '2,'
# os.chdir("hmr4d/")
from pathlib import Path
import pickle as pkl
import json
from collections import defaultdict
from tqdm import tqdm
import zipfile
import shutil

import numpy as np
np.set_printoptions(suppress=True, precision=6)
import matplotlib.pyplot as plt
from einops import einsum
from PIL import Image
import lovely_tensors as lt
lt.monkey_patch()
import imageio.v3 as iio
from smplx.joint_names import SMPLH_JOINT_NAMES
SMPLH_JOINT_NAMES = np.array(SMPLH_JOINT_NAMES[:22] + ['left_hand', 'right_hand'])
import torch

import hydra
from hydra import compose, initialize_config_module
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from hmr4d.configs import store_gvhmr  # noqa: F401

import hmr4d.model.gvhmr.gvhmr_pl_demo  # noqa: F401
from hmr4d.dataset.emdb.emdb_motion_test import EmdbSmplFullSeqDataset
from hmr4d.dataset.threedpw.threedpw_motion_test import ThreedpwSmplFullSeqDataset
from hmr4d.dataset.rich.rich_motion_test import RichSmplFullSeqDataset
from hmr4d.utils.body_model import BodyModelSMPLH, BodyModelSMPLX
from hmr4d.utils.eval.eval_utils import *
from hmr4d.utils.geo.hmr_cam import estimate_K, normalize_kp2d, create_camera_sensor
from hmr4d.utils.geo_transform import compute_cam_angvel, apply_T_on_points, move_to_start_point_face_z
from hmr4d.utils.vis.renderer import Renderer, get_global_cameras_static, get_ground_params_from_points
from hmr4d.utils.video_io_utils import get_writer
from hmr4d.utils.vis.cv2_utils import draw_bbx_xys_on_image_batch, draw_kpts_with_conf_batch

device = 'cuda'

/home/guangyu/anaconda3/envs/hmr/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/netwo

In [ ]:
smpl = BodyModelSMPLH(
    model_path="inputs/checkpoints/body_models", model_type="smpl",
    gender="neutral", num_betas=10, create_body_pose=False, 
    create_betas=False, create_global_orient=False, create_transl=False,
).to(device)
smpl_male = BodyModelSMPLH(
    model_path="inputs/checkpoints/body_models", model_type="smpl",
    gender="male", num_betas=10, create_body_pose=False, 
    create_betas=False, create_global_orient=False, create_transl=False,
).to(device)
smpl_female = BodyModelSMPLH(
    model_path="inputs/checkpoints/body_models", model_type="smpl",
    gender="female", num_betas=10, create_body_pose=False, 
    create_betas=False, create_global_orient=False, create_transl=False,
).to(device)

smplx = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="neutral", num_pca_comps=12, flat_hand_mean=False,
).to(device)
smplx_male = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="male", num_pca_comps=12, flat_hand_mean=False,
).to(device)
smplx_female = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="female", num_pca_comps=12, flat_hand_mean=False,
).to(device)

smplx2smpl = torch.load("hmr4d/utils/body_model/smplx2smpl_sparse.pt").to(device)
faces_smpl = torch.from_numpy((smpl.faces).astype("int")).unsqueeze(0).to(device)
faces_smplx = torch.from_numpy((smplx.faces).astype("int")).unsqueeze(0).to(device)
J_regressor = torch.load("hmr4d/utils/body_model/smpl_neutral_J_regressor.pt").to(device)

In [ ]:
GlobalHydra.instance().clear()
with initialize_config_module(version_base="1.3", config_module="hmr4d.configs"):
    cfg = compose(config_name="demo")
    
pipeline = instantiate(cfg.pipeline, _recursive_=False).eval().to(device)
ckpt = {k.replace("pipeline.", ""):v for k,v in torch.load(cfg.ckpt_path, "cpu")['state_dict'].items()}
pipeline.load_state_dict(ckpt, strict=True)

In [7]:
test_dataset = EmdbSmplFullSeqDataset(split=2)
root_dir = Path("/data/datasets/emdb/")
# test_dataset = ThreedpwSmplFullSeqDataset()
output_dir = Path("samples/debug/emdb_debug")
output_dir.mkdir(exist_ok=True, parents=True)
use_global = True

mean_metrics = {
    'pa_mpjpe' : 42.7, 'mpjpe' : 72.6, 'pve' : 84.2, 'accel' : 3.6,
    'waa_mpjpe' : 109.1, 'wa2_mpjpe' : 274.9, 'rte' : 1.9, 'jitter' : 16.5, 'fs' : 3.5
} # emdb
# mean_metrics = {
#     'pa_mpjpe' : 36.2, 'mpjpe' : 55.6, 'pve' : 67.2, 'accel' : 5.0,
# } # 3dpw


[02/12 20:34:33][INFO] [EMDB] Full sequence, split=2
[02/12 20:34:33][INFO] [EMDB] 25 sequences. Elapsed: 0.16s


In [8]:
for batch in test_dataset:
    vid = batch['meta']['vid']
    category, vid_name = vid.split('_')[0], '_'.join(vid.split('_')[1:])
    video_dir = root_dir / f"{category}/{vid_name}/{vid}_video.mp4"
    start = batch['meta']['vid-start-end'][0]
    if not video_dir.is_file():
        print(f"Video file not found for {vid}, skipping...")
        continue
    frame = iio.imread(video_dir, index=start, plugin="pyav")
    output_dir = video_dir.parent / "images"
    output_dir.mkdir(exist_ok=True, parents=True)
    Image.fromarray(frame).save(output_dir / f"{start:05d}.png")
        # video = iio.mimread(str(video_dir))
    # folder_dir = root_dir / f"origin/imageFiles/{vid}"
    

In [ ]:
## EMDB, 3DPW
for batch in test_dataset:
    vid = batch['meta']['vid']
    print(vid)
    torch.save(batch, output_dir / f"{vid}_batch.pt")
    
    # category, vid_name = vid.split('_')[0], '_'.join(vid.split('_')[1:])
    # video_dir = root_dir / f"{category}/{vid_name}/{vid}_video.mp4"
    # # folder_dir = root_dir / f"origin/imageFiles/{vid}"
    
    for k, v in batch['smpl_params'].items():
        batch['smpl_params'][k] = batch['smpl_params'][k].unsqueeze(0).to(device)
    for k in ['K_fullimg', 'T_w2c', 'cam_angvel', 'bbx_xys', 'f_imgseq', 'kp2d']:
        batch[k] = batch[k].unsqueeze(0).to(device)
    batch['length'] = torch.tensor([batch['length']]).to(device)
    mask = batch['mask']
    
    smpl_model = smpl_male if batch['gender'] == 'male' else smpl_female
    target_w_output = smpl_model(**{k: v[0] for k, v in batch["smpl_params"].items()})
    target_w_verts = target_w_output.vertices
    target_w_j3d = torch.matmul(J_regressor, target_w_verts)
    target_c_verts = apply_T_on_points(target_w_verts, batch["T_w2c"][0])
    target_c_j3d = apply_T_on_points(target_w_j3d, batch["T_w2c"][0])
    
    ## Calc.
    obs = normalize_kp2d(batch["kp2d"], batch["bbx_xys"])
    obs[0, ~mask] = 0

    batch_ = {
        "length": batch["length"], "obs": obs,
        "bbx_xys": batch["bbx_xys"],
        "K_fullimg": batch["K_fullimg"],
        "cam_angvel": batch["cam_angvel"],
        "f_imgseq": batch["f_imgseq"],
    }

    with torch.no_grad():
        outputs = pipeline.forward(batch_, train=False, postproc=True)
        outputs["pred_smpl_params_global"] = {k: v[0] for k, v in outputs["pred_smpl_params_global"].items()}
        outputs["pred_smpl_params_incam"] = {k: v[0] for k, v in outputs["pred_smpl_params_incam"].items()}

    smpl_camera = smplx(**outputs["pred_smpl_params_incam"])
    smpl_world = smplx(**outputs["pred_smpl_params_global"])
    pred_c_verts = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in smpl_camera.vertices])
    pred_c_j3d = einsum(J_regressor, pred_c_verts, "j v, l v i -> l j i")
    pred_ay_verts = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in smpl_world.vertices])
    pred_ay_j3d = einsum(J_regressor, pred_ay_verts, "j v, l v i -> l j i")

    batch_eval_camera = {
        "pred_j3d": pred_c_j3d, "target_j3d": target_c_j3d,
        "pred_verts": pred_c_verts, "target_verts": target_c_verts,
    }
    batch_eval_global = {
        "pred_j3d_glob": pred_ay_j3d, "target_j3d_glob": target_w_j3d,
        "pred_verts_glob": pred_ay_verts, "target_verts_glob": target_w_verts,
    }
    batch_eval = {**batch_eval_camera, **batch_eval_global}
    batch_eval = {k: v.detach().cpu() for k, v in batch_eval.items()}
    batch_eval['pred_smpl_params_incam'] = {k: v.cpu() for k, v in outputs['pred_smpl_params_incam'].items()}
    batch_eval['pred_smpl_params_global'] = {k: v.cpu() for k, v in outputs['pred_smpl_params_global'].items()}
    torch.save(batch_eval, output_dir / f"{vid}_pred_target.pt")
    
    camcoord_metrics_result = compute_camcoord_metrics(batch_eval_camera, mask=mask)
    camcoord_metrics = {}
    for k in ['pa_mpjpe', 'mpjpe', 'pve']:
        v = camcoord_metrics_result[k]
        camcoord_metrics[k] = np.array([np.nan for _ in range(batch['length'].item())])
        if sum(mask) == len(v):
            camcoord_metrics[k][mask.numpy()] = v
        
    global_metrics_result = compute_global_metrics(batch_eval_global, mask=mask) if use_global else {}
    global_metrics = {}
    for k in ['wa2_mpjpe', 'waa_mpjpe', 'rte']:
        v = global_metrics_result[k]
        global_metrics[k] = np.array([np.nan for _ in range(batch['length'].item())])
        if sum(mask) == len(v):
            global_metrics[k][mask.numpy()] = v
        
    metrics = {**camcoord_metrics, **global_metrics}
    (output_dir / f"{vid}_results.pkl").write_bytes(pkl.dumps(metrics))

    ## Get Outliers
    camcoord_outliers = topk_outlier_frames(camcoord_metrics, ['pa_mpjpe', 'mpjpe', 'pve'])
    print([x['frame'] for x in camcoord_outliers])
    global_outliers = topk_outlier_frames(global_metrics, ['wa2_mpjpe', 'waa_mpjpe']) if use_global else []
    print([x['frame'] for x in global_outliers])
    outliers = camcoord_outliers + global_outliers
    (output_dir / f"{vid}_outliers.json").write_text(json.dumps(outliers, indent=4))
    
    fig, ax = plt.subplots(3, 3, figsize=(12, 12))
    for i, (key, value) in enumerate(camcoord_metrics.items()):
        ax[i//3, i%3].set_title(f"Camera-{key}")
        ax[i//3, i%3].plot(np.arange(value.shape[0]), value)
        ax[i//3, i%3].hlines(mean_metrics[key], 0, value.shape[0]-1, colors='orange', linestyles='dashed', label='Mean')
    if use_global:
        for j, (key, value) in enumerate(global_metrics.items()):
            j += 4
            ax[j//3, j%3].set_title(f"Global-{key}")
            ax[j//3, j%3].plot(np.arange(value.shape[0]), value)
            ax[j//3, j%3].hlines(mean_metrics[key], 0, value.shape[0]-1, colors='orange', linestyles='dashed', label='Mean')

    fig.suptitle(f"Metrics for {vid}")
    fig.tight_layout()
    fig.savefig(output_dir / f"{vid}_metrics.png")
    
    # for k in camcoord_outliers[:3]:
    #     frame_k = k['frame']
    #     K = batch['K_fullimg'][0, frame_k]
    #     width, height = int(K[0,2].item() * 2), int(K[1,2].item() * 2)
    #     renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K)

    #     pred_gb_verts, pred_gb_joints = move_to_start_point_face_z(pred_ay_verts, J_regressor)
    #     target_gb_verts, target_gb_joints = move_to_start_point_face_z(target_w_verts, J_regressor)

    #     combine_frame = Image.fromarray(iio.imread(video_dir, index=frame_k))
    #     v_w, v_h = combine_frame.size
    #     origin_frame = combine_frame.crop((0, 0, v_w//2, v_h))
    #     origin_frame = origin_frame.resize((width, height))
    #     result_frame = renderer_c.render_mesh(smpl_camera.vertices[frame_k], background=np.array(origin_frame), 
    #                                         colors=[0.7, 0.7, 0.9])

    #     final_frame = Image.new('RGB', (int(v_w * 1.5), v_h))
    #     final_frame.paste(combine_frame, (0, 0))
    #     final_frame.paste(Image.fromarray(result_frame).resize((v_w//2, v_h)), (v_w, 0))
    #     final_frame.save(output_dir / f"{vid}_frame_{frame_k}.png")

In [ ]:
test_dataset = RichSmplFullSeqDataset()
output_dir = Path("samples/debug/rich_debug")
output_dir.mkdir(exist_ok=True, parents=True)

mean_metrics = {
    'pa_mpjpe' : 39.5, 'mpjpe' : 66.0, 'pve' : 74.4, 'accel' : 4.1,
    'waa_mpjpe' : 78.8, 'wa2_mpjpe' : 126.3, 'rte' : 2.4, 'jitter' : 12.8, 'fs' : 3.0
} # rich

In [ ]:
## RICH
for batch in test_dataset:
    vid = '-'.join(batch['meta']['vid'].split("/")[1:])
    print(vid)
    torch.save(batch, output_dir / f"{vid}_batch.pt")

    for k, v in batch['gt_smpl_params'].items():
        batch['gt_smpl_params'][k] = batch['gt_smpl_params'][k].unsqueeze(0).to(device)
    for k in ['K_fullimg', 'T_w2c', 'T_w2ay', 'cam_angvel', 'bbx_xys', 'f_imgseq', 'kp2d']:
        batch[k] = batch[k].unsqueeze(0).to(device)
    batch['length'] = torch.tensor([batch['length']]).to(device)

    smplx_model = smplx_male if batch['gender'] == 'male' else smplx_female
    target_w_output = smplx_model(**{k: v[0] for k, v in batch["gt_smpl_params"].items()})
    target_w_verts = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in target_w_output.vertices])
    target_c_verts = apply_T_on_points(target_w_verts, batch["T_w2c"][0])
    target_c_j3d = torch.matmul(J_regressor, target_c_verts)
    offset = target_c_j3d[..., [1, 2], :].mean(-2, keepdim=True)  # (L, 1, 3)
    target_cr_j3d = target_c_j3d - offset
    target_cr_verts = target_c_verts - offset
    # optional: ay for visual comparison
    target_ay_verts = apply_T_on_points(target_w_verts, batch["T_w2ay"][0])
    target_ay_j3d = torch.matmul(J_regressor, target_ay_verts)

    ## Calc.
    obs = normalize_kp2d(batch["kp2d"], batch["bbx_xys"])
    # obs[0, ~batch["mask"][0]] = 0

    batch_ = {
        "length": batch["length"], "obs": obs,
        "bbx_xys": batch["bbx_xys"],
        "K_fullimg": batch["K_fullimg"],
        "cam_angvel": batch["cam_angvel"],
        "f_imgseq": batch["f_imgseq"],
    }

    with torch.no_grad():
        outputs = pipeline.forward(batch_, train=False, postproc=True)
        outputs["pred_smpl_params_global"] = {k: v[0] for k, v in outputs["pred_smpl_params_global"].items()}
        outputs["pred_smpl_params_incam"] = {k: v[0] for k, v in outputs["pred_smpl_params_incam"].items()}

    smpl_camera = smplx(**outputs["pred_smpl_params_incam"])
    smpl_world = smplx(**outputs["pred_smpl_params_global"])
    pred_c_verts = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in smpl_camera.vertices])
    pred_c_j3d = einsum(J_regressor, pred_c_verts, "j v, l v i -> l j i")
    pred_ay_verts = torch.stack([torch.matmul(smplx2smpl, v_) for v_ in smpl_world.vertices])
    pred_ay_j3d = einsum(J_regressor, pred_ay_verts, "j v, l v i -> l j i")

    batch_eval_camera = {
        "pred_j3d": pred_c_j3d, "target_j3d": target_c_j3d,
        "pred_verts": pred_c_verts, "target_verts": target_c_verts,
    }
    batch_eval_global = {
        "pred_j3d_glob": pred_ay_j3d, "target_j3d_glob": target_ay_j3d,
        "pred_verts_glob": pred_ay_verts, "target_verts_glob": target_ay_verts,
    }
    batch_eval = {**batch_eval_camera, **batch_eval_global}
    batch_eval = {k: v.detach().cpu() for k, v in batch_eval.items()}
    batch_eval['pred_smpl_params_incam'] = {k: v.cpu() for k, v in outputs['pred_smpl_params_incam'].items()}
    batch_eval['pred_smpl_params_global'] = {k: v.cpu() for k, v in outputs['pred_smpl_params_global'].items()}
    torch.save(batch_eval, output_dir / f"{vid}_pred_target.pt")
    
    # camcoord_metrics = compute_camcoord_metrics(batch_eval_camera, mask=batch["mask"][0])
    # global_metrics = compute_global_metrics(batch_eval_global, mask=batch["mask"][0])
    camcoord_metrics = compute_camcoord_metrics(batch_eval_camera)
    global_metrics = compute_global_metrics(batch_eval_global)
    # global_metrics = {}
    metrics = {**camcoord_metrics, **global_metrics}
    (output_dir / f"{vid}_results.pkl").write_bytes(pkl.dumps(metrics))

    ## Get Outliers
    camcoord_outliers = topk_outlier_frames(camcoord_metrics, ['pa_mpjpe', 'mpjpe', 'pve'])
    print([x['frame'] for x in camcoord_outliers])
    global_outliers = topk_outlier_frames(global_metrics, ['wa2_mpjpe', 'waa_mpjpe'])
    # global_outliers = []
    print([x['frame'] for x in global_outliers])
    outliers = camcoord_outliers + global_outliers
    (output_dir / f"{vid}_outliers.json").write_text(json.dumps(outliers, indent=4))
    
    fig, ax = plt.subplots(3, 3, figsize=(12, 12))
    for i, (key, value) in enumerate(camcoord_metrics.items()):
        ax[i//3, i%3].set_title(f"Camera-{key}")
        ax[i//3, i%3].plot(range(value.shape[0]), value)
        ax[i//3, i%3].hlines(mean_metrics[key], 0, value.shape[0]-1, colors='orange', linestyles='dashed', label='EMDB Mean')
    for j, (key, value) in enumerate(global_metrics.items()):
        j += 4
        ax[j//3, j%3].set_title(f"Global-{key}")
        ax[j//3, j%3].plot(range(value.shape[0]), value)
        ax[j//3, j%3].hlines(mean_metrics[key], 0, value.shape[0]-1, colors='orange', linestyles='dashed', label='EMDB Mean')

    fig.suptitle(f"Metrics for {vid}")
    fig.tight_layout()
    fig.savefig(output_dir / f"{vid}_metrics.png")
    
    # for k in camcoord_outliers[:3]:
    #     frame_k = k['frame']
    #     K = batch['K_fullimg'][0, frame_k]
    #     width, height = int(K[0,2].item() * 2), int(K[1,2].item() * 2)
    #     renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=K)

    #     pred_gb_verts, pred_gb_joints = move_to_start_point_face_z(pred_ay_verts, J_regressor)
    #     target_gb_verts, target_gb_joints = move_to_start_point_face_z(target_w_verts, J_regressor)

    #     combine_frame = Image.fromarray(iio.imread(video_dir, index=frame_k))
    #     v_w, v_h = combine_frame.size
    #     origin_frame = combine_frame.crop((0, 0, v_w//2, v_h))
    #     origin_frame = origin_frame.resize((width, height))
    #     result_frame = renderer_c.render_mesh(smpl_camera.vertices[frame_k], background=np.array(origin_frame), 
    #                                         colors=[0.7, 0.7, 0.9])

    #     final_frame = Image.new('RGB', (int(v_w * 1.5), v_h))
    #     final_frame.paste(combine_frame, (0, 0))
    #     final_frame.paste(Image.fromarray(result_frame).resize((v_w//2, v_h)), (v_w, 0))
    #     final_frame.save(output_dir / f"{vid}_frame_{frame_k}.png")